# 인사 데이터 텍스트 스플릿 및 청킹

### 라이브러리 설치

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 라이브러리 선언

In [2]:
# JSON 파일 읽기/쓰기 라이브러리
import json
# 파일 경로 처리 라이브러리
from pathlib import Path
# 운영체제 환경변수 읽기
import os
# .env 파일의 환경변수를 불러오는 라이브러리
from dotenv import load_dotenv
# 텍스트 청크 분할 라이브러리
from langchain_text_splitters import RecursiveCharacterTextSplitter
# LangChain 문서 객체 (텍스트 + 메타데이터를 함께 관리)
from langchain_core.documents import Document

print('라이브러리 로딩 완료!')

C:\Users\SMT04\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


라이브러리 로딩 완료!


### 환경 설정 *** 경로 확인 필요

In [3]:
# .env 파일의 설정값을 환경변수로 불러옴
load_dotenv()

# 변환된 JSONL 파일이 있는 폴더 경로
INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../JSONL/output'))
# 처리 결과를 저장할 폴더 경로
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

# 출력 폴더가 없으면 자동으로 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 청킹 하이퍼파라미터 설정
CHUNK_SIZE    = int(os.getenv('CHUNK_SIZE',    '200'))  # 청크 크기 (글자 수 기준)
CHUNK_OVERLAP = int(os.getenv('CHUNK_OVERLAP', '50'))   # 앞뒤 청크 간 겹치는 글자 수

print(f'입력 디렉토리: {INPUT_DIR}')
print(f'출력 디렉토리: {OUTPUT_DIR}')
print('\n설정된 하이퍼파라미터:')
print(f'  청크 크기   : {CHUNK_SIZE}')
print(f'  오버랩      : {CHUNK_OVERLAP}')
print('\n설정값 영향:')
print('  - 큰 청크: 문맥 유지↑, 검색 정밀도↓')
print('  - 많은 오버랩: 문맥 단절 방지, 중복↑')

입력 디렉토리: ..\JSONL\output
출력 디렉토리: output

설정된 하이퍼파라미터:
  청크 크기   : 200
  오버랩      : 50

설정값 영향:
  - 큰 청크: 문맥 유지↑, 검색 정밀도↓
  - 많은 오버랩: 문맥 단절 방지, 중복↑


# 1. 데이터 로딩

### 1-1. JSONL 파일 로드

In [4]:
# 입력 폴더에서 .jsonl 파일 목록을 이름순으로 가져옴 (변경 이력 파일 제외)
jsonl_files = sorted(
    f for f in INPUT_DIR.glob('*.jsonl')
    if f.name != 'changes_history.jsonl'
)

if not jsonl_files:
    print(f'JSONL 파일 없음: {INPUT_DIR}')
    print('JSONL 변환 노트북을 먼저 실행해 주세요.')
else:
    doc_sets = {}
    first_sample = None

    for path in jsonl_files:
        docs = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                record = json.loads(line.strip())

                doc = Document(
                    page_content=record['embedding_text'],
                    metadata={
                        'employee_id':      record['employee_id'],
                        'employee_name':    record.get('employee_name', ''),
                        'department':       record['department'],
                        'department_level': record.get('department_level', ''),
                        'position':         record['position'],
                        'position_level':   record.get('position_level', ''),
                        'embedding_vector': record['embedding_vector'],
                        'source':           record.get('source', ''),
                        'timestamp':        record.get('timestamp', ''),
                        'changed':          record.get('changed', []),
                    }
                )
                docs.append(doc)

        doc_sets[path.stem] = docs
        print(f'  로딩: {path.name}  ({len(docs):,}건)')

        if first_sample is None:
            first_sample = docs[0]

    print(f'\n로딩 완료! 총 {len(doc_sets)}개 파일')
    print('\n첫 번째 Document 샘플:')
    print('-' * 50)
    print(f'page_content: {first_sample.page_content}')
    print(f'metadata    : {first_sample.metadata}')
    print('-' * 50)

  로딩: 급여정보_정제.jsonl  (2,000건)


  로딩: 기본인사정보_정제.jsonl  (2,000건)
  로딩: 역량성과_정제.jsonl  (2,000건)
  로딩: 통합인사정보_정제.jsonl  (2,000건)

로딩 완료! 총 4개 파일

첫 번째 Document 샘플:
--------------------------------------------------
page_content: 이름: 지준서 부서: 마케팅부 직급: 사원 연봉: 32000000 잔업시간: 10 미사용휴가일수: 14 급여은행: 우리은행 계좌번호: 1002-324-124157 4대보험가입여부: 가입
metadata    : {'employee_id': 'EMP0001', 'employee_name': '지준서', 'department': '마케팅부', 'department_level': '1', 'position': '사원', 'position_level': '1', 'embedding_vector': [], 'source': '급여정보', 'timestamp': '2026-06-07 13:24:39', 'changed': []}
--------------------------------------------------


# 2. 텍스트 정규화

### 2-1. 공백 정규화

In [5]:
print('텍스트 정규화 시작...\n')

def clean(text):
    # 앞뒤 공백 제거
    text = text.strip()
    words = text.split()
    text = ' '.join(words)
    return text


normalize_count = 0

for file_name, docs in doc_sets.items():
    for doc in docs:
        original   = doc.page_content
        normalized = clean(original)
        if original != normalized:
            normalize_count += 1
        # Document의 page_content를 정규화된 텍스트로 교체
        doc.page_content = normalized

print(f'정규화 완료!')
print(f'  정규화 적용된 레코드 수: {normalize_count:,}건')
print('\n정규화 후 샘플:')
print('-' * 50)
print(first_sample.page_content)
print('-' * 50)

텍스트 정규화 시작...

정규화 완료!
  정규화 적용된 레코드 수: 0건

정규화 후 샘플:
--------------------------------------------------
이름: 지준서 부서: 마케팅부 직급: 사원 연봉: 32000000 잔업시간: 10 미사용휴가일수: 14 급여은행: 우리은행 계좌번호: 1002-324-124157 4대보험가입여부: 가입
--------------------------------------------------


# 3. 스플릿 및 청킹

### 3-1. 텍스트 분할기 설정

In [6]:
# RecursiveCharacterTextSplitter: 구분자 우선순위에 따라 자연스럽게 분할
# chunk_size보다 짧은 텍스트는 분할하지 않고 그대로 유지
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', '!', '?', ',', ' ', '']
)

print('분할기 설정 완료!')

분할기 설정 완료!


### 3-2. 청킹 실행

In [7]:
print('텍스트 청크 분할 시작...\n')

# 파일별 청킹 결과를 저장하는 딕셔너리
chunked_sets = {}

for file_name, docs in doc_sets.items():
    # split_documents: Document 리스트를 받아 청크로 분할
    # 메타데이터(employee_id, department 등)는 자동으로 각 청크에 복사됨
    chunks = text_splitter.split_documents(docs)
    chunked_sets[file_name] = chunks

    print(f'청크 분할 결과: [{file_name}]')
    print(f'  원본 레코드 수: {len(docs):,}건')
    print(f'  생성된 청크 수: {len(chunks):,}건')
    print('\n  첫 번째 청크 샘플:')
    print('  ' + '-' * 48)
    print(f'  {chunks[0].page_content}')
    print('  ' + '-' * 48)
    print(f'  청크 메타데이터: {chunks[0].metadata}\n')

print('청킹 완료!')

텍스트 청크 분할 시작...

청크 분할 결과: [급여정보_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 2,000건

  첫 번째 청크 샘플:
  ------------------------------------------------
  이름: 지준서 부서: 마케팅부 직급: 사원 연봉: 32000000 잔업시간: 10 미사용휴가일수: 14 급여은행: 우리은행 계좌번호: 1002-324-124157 4대보험가입여부: 가입
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'employee_name': '지준서', 'department': '마케팅부', 'department_level': '1', 'position': '사원', 'position_level': '1', 'embedding_vector': [], 'source': '급여정보', 'timestamp': '2026-06-07 13:24:39', 'changed': []}



청크 분할 결과: [기본인사정보_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 4,090건

  첫 번째 청크 샘플:
  ------------------------------------------------
  이름: 지준서 부서: 마케팅부 직급: 사원 성별: 남 나이: 30 생년월일: 1997-09-19 주민등록번호: 970919-1367382 병역: 군필 입사일: 2024-05-02 근속기간: 2 학력: 대졸 출신대학: 연세대학교 학점: 3
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'employee_name': '지준서', 'department': '마케팅부', 'department_level': '1', 'position': '사원', 'position_level': '1', 'embedding_vector': [], 'source': '기본인사정보', 'timestamp': '2026-06-07 13:24:39', 'changed': []}

청크 분할 결과: [역량성과_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 2,000건

  첫 번째 청크 샘플:
  ------------------------------------------------
  이름: 지준서 부서: 마케팅부 직급: 사원 성과점수: 62 인사고과_2024: C 자격증수당여부: 비해당
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'employee_name': '지준서', 'department': '마케팅부', 'department_level': '1', 'position': '사원', 'position_level': '1', 'embedding_vector': [], 'source': '역량성과', 'timestamp':

청크 분할 결과: [통합인사정보_정제]
  원본 레코드 수: 2,000건
  생성된 청크 수: 7,035건

  첫 번째 청크 샘플:
  ------------------------------------------------
  이름: 지준서 부서: 마케팅부 직급: 사원 성별: 남 나이: 30 생년월일: 1997-09-19 주민등록번호: 970919-1367382 병역: 군필 입사일: 2024-05-02 근속기간: 2 학력: 대졸 출신대학: 연세대학교 학점: 3
  ------------------------------------------------
  청크 메타데이터: {'employee_id': 'EMP0001', 'employee_name': '지준서', 'department': '마케팅부', 'department_level': '1', 'position': '사원', 'position_level': '1', 'embedding_vector': [], 'source': '통합인사정보', 'timestamp': '2026-06-07 13:24:39', 'changed': []}

청킹 완료!


# 4. 검증

### 4-1. 데이터 소실 여부 확인

In [8]:
print('데이터 소실 여부 확인 중...')
print('-' * 50)

for file_name in doc_sets:
    original_count = len(doc_sets[file_name])
    chunk_count    = len(chunked_sets[file_name])

    # 청킹 후 건수는 같거나 많아야 함 (분할되면 늘어남)
    if chunk_count >= original_count:
        status = '정상'
    else:
        status = '소실 의심'

    print(f'  {file_name}')
    print(f'  원본 {original_count:,}건 → 청크 {chunk_count:,}건  [{status}]\n')

print('-' * 50)
print('데이터 소실 여부 확인 완료!')

데이터 소실 여부 확인 중...
--------------------------------------------------
  급여정보_정제
  원본 2,000건 → 청크 2,000건  [정상]

  기본인사정보_정제
  원본 2,000건 → 청크 4,090건  [정상]

  역량성과_정제
  원본 2,000건 → 청크 2,000건  [정상]

  통합인사정보_정제
  원본 2,000건 → 청크 7,035건  [정상]

--------------------------------------------------
데이터 소실 여부 확인 완료!


### 4-2. 메타데이터 유지 확인

In [9]:
print('메타데이터 유지 확인 중...')
print('-' * 50)

# 데이터구조정의서 기준 필수 메타데이터
required_meta = ['employee_id', 'department', 'position', 'embedding_vector']

for file_name, chunks in chunked_sets.items():
    missing_count = 0

    for chunk in chunks:
        for field in required_meta:
            if field not in chunk.metadata:
                missing_count += 1
                break

    status = '정상' if missing_count == 0 else f'누락 {missing_count}건'
    print(f'  {file_name}: [{status}]')

print('-' * 50)
print('메타데이터 유지 확인 완료!')

메타데이터 유지 확인 중...
--------------------------------------------------
  급여정보_정제: [정상]
  기본인사정보_정제: [정상]
  역량성과_정제: [정상]
  통합인사정보_정제: [정상]
--------------------------------------------------
메타데이터 유지 확인 완료!


# 5. 결과 저장

In [10]:
print('결과 저장 중...\n')

for file_name, chunks in chunked_sets.items():
    out_path = OUTPUT_DIR / f'{file_name}.jsonl'

    with open(out_path, 'w', encoding='utf-8') as f:
        for chunk in chunks:
            record = {
                'employee_id':      chunk.metadata['employee_id'],
                'employee_name':    chunk.metadata.get('employee_name', ''),
                'department':       chunk.metadata['department'],
                'department_level': chunk.metadata.get('department_level', ''),
                'position':         chunk.metadata['position'],
                'position_level':   chunk.metadata.get('position_level', ''),
                'embedding_text':   chunk.page_content,
                'embedding_vector': chunk.metadata['embedding_vector'],
                'source':           chunk.metadata.get('source', ''),
                'timestamp':        chunk.metadata.get('timestamp', ''),
                'changed':          chunk.metadata.get('changed', []),
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

    print(f'  저장: {out_path.name}  ({len(chunks):,}건)')

print('\n모든 파일 저장 완료!')

결과 저장 중...

  저장: 급여정보_정제.jsonl  (2,000건)
  저장: 기본인사정보_정제.jsonl  (4,090건)


  저장: 역량성과_정제.jsonl  (2,000건)

  저장: 통합인사정보_정제.jsonl  (7,035건)

모든 파일 저장 완료!
